<a href="https://colab.research.google.com/github/A-ghori/sklearn/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from numpy.random.mtrand import uniform
import numpy as np
import pandas as pd

# Synthatic Dataset
np.random.seed(42)
city = [
    "Kolkata",
    "Mumbai",
    "Delhi",
    "Bihar",
    "Hydrabad",
    "Lucknow",
    "Bangalore",
    "Chennai",
    "Pune",
    "Ahmedabad",
    "Jaipur",
    "Surat"
]
n = 1000

df = pd.DataFrame({
    "age" : np.random.randint(10,100,n),
    "gender" : np.random.choice(["Male","Female","Others"], n),
    "fever" : np.random.uniform(97,103,n),
    "cough" : np.random.choice(["Mild", "Low", "High"], n),
    "city" : np.random.choice(city, size=n,p=[
        0.20,  # Kolkata
        0.15,  # Mumbai
        0.12,  # Delhi
        0.05,  # Bihar
        0.08,  # Hyderabad
        0.05,  # Lucknow
        0.10,  # Bangalore
        0.07,  # Chennai
        0.06,  # Pune
        0.05,  # Ahmedabad
        0.04,  # Jaipur
        0.03   # Surat
    ]),
    "has_covid" : np.random.choice(["Yes","No"],n)
})
missing_indices = np.random.choice(
    np.arange(1000),
    size = 100,
    replace = False
)
df.loc[missing_indices, 'age'] = np.nan
df.loc[missing_indices, "city"] = np.nan
df.loc[missing_indices, 'fever'] = np.nan

In [ ]:
print(df["age"].dtype)

float64


In [ ]:
df['city'].dtype # this is object by default make it string


dtype('O')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,61.0,Others,97.571569,High,Mumbai,No
1,24.0,Male,100.912739,High,Mumbai,Yes
2,81.0,Male,98.872888,High,Bihar,Yes
3,70.0,Female,99.394276,Mild,Mumbai,Yes
4,30.0,Female,102.147918,High,Mumbai,Yes


In [ ]:
from sklearn.impute import SimpleImputer # used for fill the missing values with NaN
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df['age'].isna().sum()

In [ ]:
df['city'].isna().sum()

In [ ]:
df['city'].head()
df['city'].head(50)
df['city'].value_counts()

## For Cover those Null values in the age col

In [ ]:
imputer_age = SimpleImputer(strategy="mean")
df["age"] = imputer_age.fit_transform(df[["age"]]).ravel() # fit -> calculate cols mean tranform -> NaN ko us mean se replace karega and ravel -> make the array into 1-D
df["age"].isna().sum()


## For cover those NaN values in the city cols


In [ ]:
imputer_city = SimpleImputer(strategy="most_frequent")

df["city"] = imputer_city.fit_transform(
    df[["city"]]
).ravel()

df['city'].isna().sum()

In [ ]:
df["city"].head(50)

In [ ]:
df.isnull().sum()

# Fever Null values replace by mean

In [ ]:
imputer_fever = SimpleImputer(strategy="mean")
df["fever"] = imputer_fever.fit_transform(df[["fever"]]).ravel()


In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["has_covid"])
y = df["has_covid"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2 ,random_state=42)

In [ ]:
## Extracting Age

X_train_age = X_train.drop(columns=['city','gender','fever','cough']).values
X_test_age = X_test.drop(columns=['city','gender','fever','cough']).values

X_train_age


In [ ]:
X_train

In [ ]:
y_train

## Gender -> One Hot
## Fever -> Simple Imputer
## Cough -> Ordinal
## City -> One Hot
## Age -> Standardization / Feature Scaling

In [ ]:
# AGe Standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_age = scaler.fit_transform(X_train[["age"]])
X_test_age = scaler.transform(X_test[["age"]])

X_train_age


In [ ]:
X_test.isnull().sum()

## Ordinal -> Cough

In [ ]:
oe = OrdinalEncoder(categories=[["Mild", "Low", "High"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])
X_test_cough = oe.transform(X_test[["cough"]])

X_train_cough.shape

## One Hot Encoder -> City , gender

In [ ]:
result = pd.get_dummies(df, columns=['city','gender'])
result.shape
result.head()

In [ ]:
ohe = OneHotEncoder(drop="first")
X_train_gender_city = ohe.fit_transform(X_train[["city","gender"]])
X_test_gender_city = ohe.transform(X_test[["city","gender"]])

X_test_gender_city.shape
X_train_gender_city


In [ ]:
X_train_gender_city = ohe.fit_transform(
    X_train[['gender', 'city']]
).toarray()

X_test_gender_city = ohe.transform(
    X_test[['gender', 'city']]
).toarray()
X_train_gender_city

In [ ]:
from sklearn.impute import SimpleImputer

imputer_fever = SimpleImputer(strategy="mean")

X_train_fever = imputer_fever.fit_transform(X_train[['fever']])
X_test_fever = imputer_fever.transform(X_test[['fever']])

In [ ]:
print(X_train_age.shape)
print(X_train_gender_city.shape)
print(X_train_cough.shape)
print(X_train_fever.shape)

In [ ]:
print(type(X_train_age), X_train_age.shape)
print(type(X_train_gender_city), X_train_gender_city.shape)
print(type(X_train_cough), X_train_cough.shape)
print(type(X_train_fever), X_train_fever.shape)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_gender_city, X_train_cough, X_train_fever),axis=1)
X_train_transformed.shape

In [ ]:
X_test_transformed = np.concatenate((X_test_age,X_test_gender_city, X_test_cough, X_test_fever),axis=1)

X_test_transformed.shape

## Now using Column Transformer

In [30]:
from sklearn.compose import ColumnTransformer

In [36]:
transformer = ColumnTransformer(transformers=[
  ("tnf1",SimpleImputer(), ["fever"]),
  ("tnf2", OrdinalEncoder(categories=[["Mild","Low","High"]]), ["cough"]) ,
  ("tnf3", OneHotEncoder(sparse_output=False, drop="first"), ["gender","city"]),
  ("tnf4",StandardScaler(), ["age"])
],remainder="passthrough")

X_train_Tranformer=transformer.fit_transform(X_train)
X_test_Transformer=transformer.transform(X_test)

In [38]:
X_test_Transformer.shape

(200, 16)

In [37]:
X_train_Tranformer.shape

(800, 16)